# B — SSVI Parameter Dynamics: Time-Series Modelling & Surface Forecasting

**Politecnico di Milano — Insurance & Econometrics (A.Y. 2025/26)**  
Authors: Alessio Porrini, Marco Amarilli, Camilla Introzzi, Christian Frigerio

---

## Overview

This notebook provides a unified time-series analysis of the five SSVI parameters
(α, β, ρ, η, γ) calibrated daily on SPX options 2010–2020 (~2,666 trading days).
All data are loaded directly from the project GitHub repository.

**Sections:**
1. Data loading and descriptive overview  
2. Stationarity analysis (ADF + KPSS on levels **and** first differences)  
3. ARMA modelling — all series, levels and differences  
4. ARMAX modelling — no-leakage framework (X_t → y_{t+1})  
5. AR-GARCH prediction intervals  
6. VAR analysis — correct random-walk baseline  
7. Johansen cointegration + VECM  
8. Rolling Johansen — regime-conditional cointegration  
9. PCA on the full SSVI surface + HAR on PC scores  
10. Comprehensive results summary  

---

**Key references:**  
- Gatheral & Jacquier (2014). Arbitrage-free SVI volatility surfaces. *Quantitative Finance*, 14(1), 59–71.  
- Corsi (2009). A simple approximate long-memory model of realized volatility. *J. Financial Econometrics*, 7(2), 174–196.  
- Johansen (1991). Estimation and hypothesis testing of cointegration vectors. *Econometrica*, 59(6), 1551–1580.  
- Engle & Granger (1987). Co-integration and error correction. *Econometrica*, 55(2), 251–276.  
- Bollerslev & Wooldridge (1992). Quasi-maximum likelihood estimation. *Econometric Reviews*, 11(2), 143–172.  
- Andrès, Boumezoued & Jourdain (2025). The implied volatility surface (also) is path-dependent. arXiv v3.  
- Diebold & Mariano (1995). Comparing predictive accuracy. *J. Business & Economic Statistics*, 13(3), 253–263.

In [ ]:
import warnings, os
import numpy as np
import pandas as pd
from pathlib import Path
from itertools import product
from scipy import stats

import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns

import statsmodels.api as sm
from statsmodels.tsa.stattools import adfuller, kpss, grangercausalitytests as gc_test
from statsmodels.tsa.arima.model import ARIMA
from statsmodels.tsa.vector_ar.var_model import VAR
from statsmodels.stats.diagnostic import acorr_ljungbox, het_arch
from statsmodels.stats.outliers_influence import variance_inflation_factor
from statsmodels.tools.tools import add_constant

from sklearn.decomposition import PCA
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import StandardScaler

try:
    from statsmodels.tsa.vector_ar.vecm import VECM, select_coint_rank
    HAS_VECM = True
except ImportError:
    HAS_VECM = False

try:
    from arch import arch_model as arch_fn
    HAS_ARCH = True
except ImportError:
    HAS_ARCH = False
    print('arch not installed: pip install arch')

try:
    import pandas_datareader.data as web
    HAS_PDR = True
except ImportError:
    HAS_PDR = False
    print('pandas-datareader not installed: pip install pandas-datareader')

warnings.filterwarnings('ignore')
pd.set_option('display.float_format', '{:.4f}'.format)
pd.set_option('display.max_columns', 20)
pd.set_option('display.width', 160)
plt.rcParams.update({'figure.dpi': 110, 'axes.grid': True, 'grid.alpha': 0.3,
                     'axes.spines.top': False, 'axes.spines.right': False})

OUTPUT_DIR = Path('../output')
PLOT_DIR   = OUTPUT_DIR / 'figures'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
PLOT_DIR.mkdir(parents=True, exist_ok=True)

GITHUB   = 'https://raw.githubusercontent.com/aporrini/Econometrics-Volatility-Surface-Dynamics/main/Data'
START    = pd.Timestamp('2010-01-04')
END      = pd.Timestamp('2020-12-31')
SPLIT    = 0.80
PARAMS   = ['alpha', 'beta', 'rho', 'eta', 'gamma']
D_PARAMS = ['d_alpha', 'd_beta', 'd_rho', 'd_eta', 'd_gamma']
Z80      = 1.2816
RANDOM_STATE = 42
print('Setup complete.')

In [ ]:
import sys
sys.path.insert(0, '../src')

from ssvi_helpers       import ssvi_omega, ssvi_atm_iv
from stats_helpers      import chow_test, dm_hln
from scoring            import pinball_loss, winkler_score, coverage as pi_coverage
from forecasting_helpers import smooth_stress_indicator

print('src/ helpers imported.')

---
## 1. Data Loading

In [ ]:
# --- SSVI calibration results from GitHub ---
print('Loading SSVI data from GitHub...')
ssvi_raw = pd.read_csv(f'{GITHUB}/ssvi_all_dates_clean_results.csv')
ssvi = ssvi_raw[ssvi_raw['success'] == True].copy()
ssvi['date'] = START + pd.to_timedelta(ssvi['time_elapsed'], unit='D')
ssvi = (ssvi[['date', 'time_elapsed', 'alpha', 'beta', 'rho', 'eta', 'gamma', 'rmse_iv', 'n_obs']]
        .sort_values('date').reset_index(drop=True))
print(f'  SSVI: {len(ssvi):,} successful calibrations | {ssvi["date"].min().date()} to {ssvi["date"].max().date()}')

# --- No-arbitrage diagnostics from GitHub ---
na_raw = pd.read_csv(f'{GITHUB}/no_arbitrage_clean_results.csv')
ssvi = ssvi.merge(na_raw[['time_elapsed', 'max_cond1', 'max_cond2',
                           'butterfly_ok', 'calendar_ok']],
                  on='time_elapsed', how='left')
print(f'  No-arb: {len(na_raw):,} rows merged.')

In [ ]:
# --- Market data from FRED (pandas_datareader) ---
# VIX daily: CBOE Volatility Index via FRED series VIXCLS
mkt = {}
if HAS_PDR:
    print('Downloading VIX from FRED (series: VIXCLS)...')
    try:
        vix_fred = web.DataReader('VIXCLS', 'fred',
                                   start=START - pd.Timedelta(days=30),
                                   end=END + pd.Timedelta(days=10))
        vix_s = vix_fred.squeeze().rename('vix')
        vix_s.index = pd.to_datetime(vix_s.index).normalize()
        mkt['vix'] = vix_s.resample('D').ffill().loc[START:END]
        print(f'  VIX (VIXCLS): {len(mkt["vix"]):,} obs  '
              f'mean={mkt["vix"].mean():.2f}  max={mkt["vix"].max():.2f}')
    except Exception as e:
        print(f'  VIX (VIXCLS) download failed: {e}')
else:
    print('pandas-datareader not available — install with: pip install pandas-datareader')

# Build mdf from dict of Series; normalize index name BEFORE reset_index
# so the column is always 'date' regardless of what FRED names its index ('DATE', etc.)
mdf = pd.DataFrame(mkt)
mdf.index = pd.to_datetime(mdf.index)
mdf.index.name = 'date'
mdf = mdf.reset_index()   # now 'date' column is guaranteed

# Derived features (all backward-looking, no leakage)
if 'vix' in mdf.columns:
    mdf['log_vix']    = np.log(mdf['vix'].clip(lower=0.01))
    mdf['d_log_vix']  = mdf['log_vix'].diff()
    mdf['vix_zscore'] = ((mdf['vix'] - mdf['vix'].rolling(21).mean())
                         / (mdf['vix'].rolling(21).std() + 1e-9))

# Merge with SSVI dataset
mkt_cols = ['date'] + [c for c in mdf.columns if c != 'date']
df = ssvi.merge(mdf[mkt_cols], on='date', how='left').sort_values('date').reset_index(drop=True)
df = df.ffill()

# First differences of all 5 SSVI parameters
for p in PARAMS:
    df[f'd_{p}'] = df[p].diff()

df = df.dropna(subset=D_PARAMS).reset_index(drop=True)
print(f'\nFinal dataset: {len(df):,} obs | {df["date"].min().date()} to {df["date"].max().date()}')

# Temporal 80/20 split
n_split  = int(len(df) * SPLIT)
df_train = df.iloc[:n_split].copy()
df_test  = df.iloc[n_split:].copy()
print(f'Train: {len(df_train):,} ({df_train["date"].min().date()} → {df_train["date"].max().date()})')
print(f'Test : {len(df_test):,} ({df_test["date"].min().date()} → {df_test["date"].max().date()})')

In [ ]:
# --- Descriptive statistics ---
print('=== SSVI Parameter Descriptive Statistics (full sample) ===')
desc = df[PARAMS].describe().round(4)
print(desc.to_string())

print('\n=== Correlation matrix ===')
print(df[PARAMS].corr().round(3).to_string())

# Time-series overview plot
fig, axes = plt.subplots(3, 2, figsize=(14, 10), constrained_layout=True)
fig.suptitle('SSVI Parameters — Full Sample 2010–2020', fontsize=12)
split_date = df_test['date'].iloc[0]

plot_series = [('alpha', r'$\alpha$ (log ATM level)'),
               ('beta',  r'$\beta$ (term-structure slope)'),
               ('rho',   r'$\rho$ (skew / leverage)'),
               ('eta',   r'$\eta$ (vol-of-vol / curvature)'),
               ('gamma', r'$\gamma$ (wing asymmetry)')]

for ax, (col, lab) in zip(axes.flat[:5], plot_series):
    ax.plot(df['date'], df[col], lw=0.8, color='steelblue', alpha=0.6)
    ax.plot(df['date'], df[col].rolling(20, center=True).mean(), lw=1.8, color='steelblue')
    ax.axvline(split_date, color='crimson', lw=1.2, ls='--', label='Train/Test')
    ax.set_ylabel(lab, fontsize=9)
    ax.legend(fontsize=7)

axes.flat[5].set_visible(False)
plt.savefig(PLOT_DIR / 'B_ssvi_parameters_timeseries.png', dpi=130, bbox_inches='tight')
plt.show()
print('Saved: figures/B_ssvi_parameters_timeseries.png')

---
## 2. Stationarity Analysis

We test all **10 series** (5 levels + 5 first differences) for unit roots using:

- **ADF test** (Dickey & Fuller 1979): H0 = unit root (non-stationary). Automatic BIC lag selection.
- **KPSS test** (Kwiatkowski et al. 1992): H0 = stationarity. Complementary to ADF.

**Decision rule** (conflicting tests → I(d) uncertain):

| ADF rejects H0 | KPSS does NOT reject H0 | Conclusion |
|:-:|:-:|:-:|
| Yes | Yes | **I(0)** — stationary |
| No | No | **I(1)** — unit root |
| Conflicting | — | Uncertain — inspect ACF |

This determines whether each series should enter ARMA **in levels** (I(0)) or **in first differences** (I(1)).
Per the user's request, ARMA/ARMAX are also run on the differenced version of I(0) series for comparison.

In [ ]:
def adf_test(series, label):
    y = series.dropna().values
    res = adfuller(y, autolag='BIC', maxlag=20)
    reject = res[1] < 0.05
    return {'series': label, 'adf_stat': round(res[0], 4), 'p_value': round(res[1], 4),
            'lags': res[2], 'crit_5pct': round(res[4]['5%'], 4),
            'reject_H0': reject, 'conclusion_ADF': 'I(0)' if reject else 'I(1)+'}

def kpss_test(series, label):
    y = series.dropna().values
    stat, p, lags, crit = kpss(y, regression='c', nlags='auto')
    reject = p < 0.05  # reject stationarity
    return {'series': label, 'kpss_stat': round(stat, 4), 'p_value': round(p, 4),
            'lags': lags, 'reject_H0': reject, 'conclusion_KPSS': 'non-stat' if reject else 'stat'}

ALL_SERIES = PARAMS + D_PARAMS
adf_rows  = [adf_test(df[s], s)  for s in ALL_SERIES]
kpss_rows = [kpss_test(df[s], s) for s in ALL_SERIES]
adf_df    = pd.DataFrame(adf_rows)
kpss_df   = pd.DataFrame(kpss_rows)

# Merge and determine integration order
unit_root = adf_df.merge(kpss_df[['series','kpss_stat','p_value','conclusion_KPSS']],
                          on='series', suffixes=('_ADF', '_KPSS'))

def integration_order(row):
    adf_ok   = row['reject_H0']            # True = ADF rejects H0 → stationary
    kpss_nst = row['conclusion_KPSS'] == 'non-stat'  # True = non-stationary
    if adf_ok and not kpss_nst:
        return 'I(0)'
    elif not adf_ok and kpss_nst:
        return 'I(1)'
    else:
        return 'Uncertain'

unit_root['I_order'] = unit_root.apply(integration_order, axis=1)

display_cols = ['series', 'adf_stat', 'p_value_ADF', 'conclusion_ADF',
                'kpss_stat', 'p_value_KPSS', 'conclusion_KPSS', 'I_order']
print('=== Stationarity Test Results ===')
print(unit_root[display_cols].to_string(index=False))
unit_root[display_cols].to_csv(OUTPUT_DIR / 'B_stationarity_tests.csv', index=False)
print('\nSaved: output/B_stationarity_tests.csv')

# Determine correct specification for ARMA
integration = {}
for p in PARAMS:
    row = unit_root[unit_root['series'] == p].iloc[0]
    integration[p] = row['I_order']
print('\n=== Integration Orders (level series) ===')
for k, v in integration.items():
    print(f'  {k:<8}: {v}')

---
## 3. ARMA Modelling — All Series (Levels and Differences)

**Specification strategy:**
- I(1) series (α, η): ARMA on **first differences** Δα, Δη (correct specification).
- I(0) series (β, ρ, γ): ARMA on **levels** (theoretically correct) **and** on **first differences** for completeness.

**Order selection:** BIC grid over ARMA(p,q) for p,q ∈ {0,1,2,3}, estimated on the training set.  
**OOS evaluation:** rolling expanding window, h=1 step ahead, MSE ratio = MSE_model / MSE_random_walk.  
**In-sample diagnostics:** Ljung-Box (serial correlation), ARCH-LM (conditional heteroscedasticity),  
Jarque-Bera (normality). ARCH effects → QMLE / GARCH residuals warranted (Bollerslev & Wooldridge 1992).

In [ ]:
def select_arma_order(y_train, max_p=3, max_q=3):
    """BIC grid search for ARMA(p,q) order on training data."""
    best_bic = np.inf
    best_pq  = (1, 0)
    y = y_train.dropna().values
    for p, q in product(range(max_p + 1), range(max_q + 1)):
        if p == 0 and q == 0:
            continue
        try:
            m = ARIMA(y, order=(p, 0, q), trend='c').fit()
            if m.bic < best_bic:
                best_bic = m.bic
                best_pq  = (p, q)
        except:
            pass
    return best_pq, round(best_bic, 2)

def rolling_arma_oos(y_full, p, q, n_train):
    """Rolling expanding-window OOS, h=1, vs random-walk baseline."""
    y = np.asarray(y_full.dropna())
    N = len(y)
    preds = []
    for i in range(N - n_train):
        y_tr = y[:n_train + i]
        try:
            m  = ARIMA(y_tr, order=(p, 0, q), trend='c').fit()
            fc = float(m.forecast(steps=1).iloc[0])
        except:
            fc = float(y_tr[-1])
        preds.append(fc)
    actual   = y[n_train:]
    baseline = y[n_train - 1:-1]   # random walk: y_{t-1}
    preds    = np.array(preds)
    mse_m = float(np.mean((actual - preds) ** 2))
    mse_b = float(np.mean((actual - baseline) ** 2))
    ratio = mse_m / mse_b if mse_b > 1e-15 else np.nan
    r2    = float(1 - mse_m / mse_b) if mse_b > 1e-15 else np.nan
    return {'ratio': ratio, 'r2_oos': r2, 'mse_m': mse_m, 'mse_b': mse_b,
            'preds': preds, 'actual': actual, 'baseline': baseline}

def arma_insample_diag(y_train, p, q):
    """Fit ARMA on training data and return in-sample diagnostics."""
    y = y_train.dropna().values
    try:
        m = ARIMA(y, order=(p, 0, q), trend='c').fit()
        resid = m.resid
        lb_p  = float(acorr_ljungbox(resid, lags=[10], return_df=True)['lb_pvalue'].iloc[0])
        arch_p = np.nan
        try:
            _, arch_p, _, _ = het_arch(resid, nlags=5)
            arch_p = float(arch_p)
        except:
            pass
        jb_p  = float(stats.jarque_bera(resid)[1])
        return {'lb_p': round(lb_p, 4), 'arch_p': round(arch_p, 4) if not np.isnan(arch_p) else np.nan,
                'jb_p': round(jb_p, 4), 'ok': lb_p > 0.05}
    except:
        return {'lb_p': np.nan, 'arch_p': np.nan, 'jb_p': np.nan, 'ok': False}

print('ARMA order selection (BIC, training set)...')
arma_orders = {}
for s in ALL_SERIES:
    order, bic = select_arma_order(df_train[s], max_p=3, max_q=3)
    arma_orders[s] = order
    print(f'  {s:<15}: ARMA{order}  BIC={bic}')

In [ ]:
print('Rolling OOS evaluation (expanding window, h=1)...')
arma_results = {}
for s in ALL_SERIES:
    p, q = arma_orders[s]
    res  = rolling_arma_oos(df[s].dropna(), p, q, n_split)
    diag = arma_insample_diag(df_train[s], p, q)
    arma_results[s] = {**res, **diag, 'p': p, 'q': q}
    lb_flag = 'OK' if diag['ok'] else 'SERIAL-CORR'
    print(f'  {s:<15}: MSE-ratio={res["ratio"]:.4f}  R2_OOS={res["r2_oos"]:.4f}  '
          f'LB_p={diag["lb_p"]:.3f} {lb_flag}')

# Summary table
rows = []
for s in ALL_SERIES:
    r = arma_results[s]
    rows.append({'series': s, 'order': f'ARMA({r["p"]},{r["q"]})',
                 'MSE_ratio': round(r['ratio'], 4), 'R2_OOS': round(r['r2_oos'], 4),
                 'LB_p': r['lb_p'], 'ARCH_p': r['arch_p'], 'JB_p': r['jb_p'],
                 'LB_adequate': r['ok']})

arma_df = pd.DataFrame(rows)
print('\n=== ARMA Summary Table ===')
print(arma_df.to_string(index=False))
arma_df.to_csv(OUTPUT_DIR / 'B_arma_results.csv', index=False)
print('\nSaved: output/B_arma_results.csv')

print('\nNote: ARCH_p < 0.05 → GARCH residuals warranted (QMLE standard errors apply).')
print('Note: JB_p << 0.05 → non-normal residuals (expected in finance; QMLE still consistent).')

In [ ]:
# --- OOS forecast tracking plot for key series ---
focus = ['d_alpha', 'd_eta', 'rho', 'gamma']
fig, axes = plt.subplots(len(focus), 2, figsize=(14, 4 * len(focus)), constrained_layout=True)
fig.suptitle('ARMA OOS h=1: Actual vs Predicted (right: cumulative squared error)', fontsize=11)

for ri, s in enumerate(focus):
    if s not in arma_results:
        continue
    r      = arma_results[s]
    actual = r['actual']
    preds  = r['preds']
    bline  = r['baseline']
    xs     = np.arange(len(actual))

    axes[ri, 0].plot(xs, actual, 'k-', lw=0.8, label='Actual', alpha=0.8)
    axes[ri, 0].plot(xs, preds,  'b-', lw=1.2, label=f'ARMA{arma_orders[s]}', alpha=0.75)
    axes[ri, 0].plot(xs, bline,  '--', color='gray', lw=0.7, alpha=0.5, label='RW baseline')
    axes[ri, 0].set_title(f'{s}  MSE-ratio={r["ratio"]:.4f}  R2={r["r2_oos"]:.4f}', fontsize=9)
    axes[ri, 0].legend(fontsize=7, ncol=3)

    cum_m = np.cumsum((actual - preds) ** 2)
    cum_b = np.cumsum((actual - bline) ** 2)
    axes[ri, 1].plot(xs, cum_m, 'b-', lw=1.3, label='ARMA')
    axes[ri, 1].plot(xs, cum_b, '--', color='gray', lw=1.0, label='RW baseline')
    axes[ri, 1].fill_between(xs, cum_m, cum_b, where=cum_m < cum_b, alpha=0.15, color='blue')
    axes[ri, 1].set_title(f'{s} — cumulative squared error', fontsize=9)
    axes[ri, 1].legend(fontsize=7)

plt.savefig(PLOT_DIR / 'B_arma_oos_forecast.png', dpi=130, bbox_inches='tight')
plt.show()
print('Saved: figures/B_arma_oos_forecast.png')

---
## 4. ARMAX Modelling — No-Leakage Framework

**Design:** features at time $t$ (X_t) predict the parameter at time $t+1$ — genuine forecasting, not nowcasting.  
X is lagged by exactly one day before entering the ARMAX: $\hat{y}_{t+1} = f(y_t, \ldots, X_t)$.

**Exogenous features (all I(0), sourced from FRED):**
- `d_log_vix`: Δlog(VIX) — daily change in implied vol level (CBOE VIX via FRED VIXCLS)
- `vix_zscore`: VIX standardised deviation from rolling 21-day mean

**Feature selection:** Granger causality pre-screening + VIF collinearity check on **training set only**
(no leakage of test information into feature selection).  
**Evaluation:** rolling expanding-window OOS, h=1, MSE ratio vs random-walk baseline.  
**DM-HLN test:** Diebold-Mariano with HLN correction; positive DM stat = ARMAX beats RW.

In [ ]:
# Available exogenous features (only FRED-sourced variables)
EXOG_CANDIDATES = [c for c in ['d_log_vix', 'vix_zscore']
                   if c in df.columns and df[c].notna().sum() > 500]
print(f'Available exogenous features: {EXOG_CANDIDATES}')

if not EXOG_CANDIDATES:
    print('No FRED features available — ARMAX will be skipped.')
    selected_feat = []
else:
    # Granger causality pre-screening on TRAINING SET ONLY
    gr_rows = []
    for target in ALL_SERIES:
        for feat in EXOG_CANDIDATES:
            sub = df_train[[target, feat]].dropna()
            if len(sub) < 40:
                continue
            try:
                res = gc_test(sub, maxlag=2, verbose=False)
                p1  = res[1][0]['ssr_ftest'][1]
                gr_rows.append({'target': target, 'feature': feat,
                                'p_lag1': round(p1, 4), 'sig': p1 < 0.10})
            except:
                pass

    gr_df = pd.DataFrame(gr_rows)
    print('\nGranger-significant pairs (p < 0.10, training set only):')
    if len(gr_df) > 0:
        print(gr_df[gr_df['sig']].sort_values(['target', 'p_lag1']).to_string(index=False))
    else:
        print('  No pairs tested.')

    # VIF check on selected features
    feat_pool = list(dict.fromkeys(
        [f for t in ALL_SERIES for f in
         gr_df[(gr_df['target'] == t) & gr_df['sig']]['feature'].tolist()
         if f in df_train.columns]
    )) if len(gr_df) > 0 else []
    feat_pool = [f for f in EXOG_CANDIDATES if f in feat_pool]
    if not feat_pool:
        feat_pool = EXOG_CANDIDATES
        print('No Granger-significant features; using all available.')

    if len(feat_pool) >= 2:
        Xv   = add_constant(df_train[feat_pool].dropna())
        vifs = [variance_inflation_factor(Xv.values, i + 1) for i in range(len(feat_pool))]
        vif_df = pd.DataFrame({'feature': feat_pool, 'VIF': vifs}).sort_values('VIF', ascending=False)
        print('\nVIF (train):')
        print(vif_df.round(2).to_string(index=False))
        selected_feat = vif_df[vif_df['VIF'] < 10]['feature'].tolist()
    else:
        selected_feat = feat_pool

    print(f'\nSelected features ({len(selected_feat)}): {selected_feat}')

In [ ]:
def dm_test(actual, pred_a, pred_b, h=1):
    """Diebold-Mariano test (Newey-West HAC, HLN small-sample correction)."""
    e_a = np.asarray(actual) - np.asarray(pred_a)
    e_b = np.asarray(actual) - np.asarray(pred_b)
    d   = e_a ** 2 - e_b ** 2
    n   = len(d)
    dbar = d.mean()
    lag  = max(1, h - 1)
    g0   = np.mean((d - dbar) ** 2)
    acov = sum((1 - k / (lag + 1)) * np.mean((d[k:] - dbar) * (d[:-k] - dbar))
               for k in range(1, lag + 1))
    v = (g0 + 2 * acov) / n
    if v <= 1e-15:
        return np.nan, np.nan
    dm_raw  = dbar / np.sqrt(v)
    hln_fac = np.sqrt((n + 1 - 2 * h + h * (h - 1) / n) / n)
    dm_hln  = dm_raw * hln_fac
    p = float(2 * stats.t.sf(abs(dm_hln), df=n - 1))
    return round(float(dm_hln), 4), round(p, 4)

def rolling_armax_oos(y_full, X_full, p, q, n_train):
    """Rolling OOS for ARMAX(p,q). X is LAGGED by 1 (X_t predicts y_{t+1})."""
    y = np.asarray(y_full.dropna())
    N = min(len(y), len(X_full))
    y, X = y[:N], X_full[:N]
    preds = []
    for i in range(N - n_train):
        y_tr = y[:n_train + i]
        X_tr = X[:n_train + i]
        X_te = X[n_train + i: n_train + i + 1]  # X_t used to predict y_{t+1}
        try:
            m  = ARIMA(y_tr, exog=X_tr, order=(p, 0, q), trend='c').fit()
            fc = float(m.forecast(steps=1, exog=X_te).iloc[0])
        except:
            fc = float(y_tr[-1])
        preds.append(fc)
    actual   = y[n_train:]
    baseline = y[n_train - 1:-1]
    preds    = np.array(preds)
    mse_m = float(np.mean((actual - preds) ** 2))
    mse_b = float(np.mean((actual - baseline) ** 2))
    ratio = mse_m / mse_b if mse_b > 1e-15 else np.nan
    r2    = float(1 - mse_m / mse_b) if mse_b > 1e-15 else np.nan
    return {'ratio': ratio, 'r2_oos': r2, 'preds': preds, 'actual': actual, 'baseline': baseline}

print('Running ARMAX OOS (no-leakage: X_t -> y_{t+1})...')
armax_results = {}

if selected_feat:
    mu_x = df_train[selected_feat].mean()
    sd_x = df_train[selected_feat].std().replace(0, 1)

    df_exog = df[selected_feat].copy()
    df_exog_sc = ((df_exog - mu_x) / sd_x).fillna(0)
    # LAG by 1: X used at time t predicts y at time t+1
    X_lagged = df_exog_sc.shift(1).fillna(0).values

    for s in ALL_SERIES:
        p, q = arma_orders[s]
        y_s  = df[s].dropna()
        N    = min(len(y_s), len(X_lagged))
        res  = rolling_armax_oos(y_s[:N], X_lagged[:N], p, q, n_split)
        dm_stat, dm_p = dm_test(res['actual'], res['preds'], res['baseline'], h=1)
        armax_results[s] = {**res, 'dm_stat': dm_stat, 'dm_p': dm_p}
        sig = '**' if dm_p < 0.05 else ('.' if dm_p < 0.10 else '') if not np.isnan(dm_p) else ''
        print(f'  {s:<15}: MSE-ratio={res["ratio"]:.4f}  R2={res["r2_oos"]:.4f}  '
              f'DM={dm_stat}  p={dm_p} {sig}')
else:
    print('No features selected; ARMAX skipped.')

In [ ]:
# ARMAX summary table
if armax_results:
    armax_rows = []
    for s in ALL_SERIES:
        r = armax_results[s]
        armax_rows.append({
            'series':     s,
            'order':      f'ARMAX{arma_orders[s]}',
            'MSE_ratio':  round(r['ratio'], 4),
            'R2_OOS':     round(r['r2_oos'], 4),
            'DM_stat':    r['dm_stat'],
            'DM_p':       r['dm_p'],
            'beats_RW':   r['ratio'] < 1.0
        })
    armax_df = pd.DataFrame(armax_rows)
    print('=== ARMAX Summary (no-leakage: X_t -> y_{t+1}) ===')
    print(armax_df.to_string(index=False))
    armax_df.to_csv(OUTPUT_DIR / 'B_armax_results.csv', index=False)
    print('\nSaved: output/B_armax_results.csv')

    # Compare ARMA vs ARMAX side by side
    compare_rows = []
    for s in ALL_SERIES:
        ar = arma_results.get(s, {})
        ax = armax_results.get(s, {})
        compare_rows.append({
            'series':       s,
            'ARMA_ratio':   round(ar.get('ratio', np.nan), 4),
            'ARMAX_ratio':  round(ax.get('ratio', np.nan), 4),
            'gain':         round(ar.get('ratio', 1.0) - ax.get('ratio', 1.0), 4),
        })
    print('\n=== ARMA vs ARMAX MSE-ratio (gain > 0: ARMAX improves on ARMA) ===')
    print(pd.DataFrame(compare_rows).to_string(index=False))
else:
    print('ARMAX skipped (no features available).')

---
## 5. AR-GARCH Prediction Intervals

SSVI parameters exhibit ARCH effects (Engle 1982): squared residuals are serially correlated,
meaning the conditional variance is time-varying.  
An **AR(1)-GARCH(1,1)** model accounts for this, with QMLE standard errors
(Bollerslev & Wooldridge 1992) for robustness to non-Gaussian innovations.

We target **80% prediction intervals** (coverage = 0.80, Z₈₀ = 1.2816) and evaluate three scoring rules:

| Score | Formula | Interpretation |
|-------|---------|----------------|
| **Coverage** | Fraction of $y_t \in [L_t, U_t]$ | Calibration: should be ≈ 0.80 |
| **Winkler score** (Winkler 1972) | Width + penalty for violations | Sharpness: lower = better |
| **Pinball loss** (Koenker & Bassett 1978) | $\rho_\tau(y - q) = \max(\tau(y-q), (\tau-1)(y-q))$ | Proper quantile score: lower = better |

Pinball is computed at τ=0.10 (lower bound) and τ=0.90 (upper bound); reported as their average.  
Pinball is a proper scoring rule: a model cannot game it by widening the interval indefinitely.

In [ ]:
# pinball_loss, winkler_score, pi_coverage imported from src/scoring.py

if not HAS_ARCH:
    print('arch library not available; skipping GARCH section. pip install arch')
else:
    print('AR(1)-GARCH(1,1) Prediction Intervals (80% target, QMLE)...')
    pi_rows = []
    PI_SERIES = [s for s in ALL_SERIES if s in PARAMS or s in ['d_alpha', 'd_eta']]

    for s in PI_SERIES:
        y_tr = df_train[s].dropna().values
        y_te = df_test[s].dropna().values
        n_te = len(y_te)
        try:
            am = arch_fn(y_tr, mean='AR', lags=1, vol='GARCH', p=1, q=1, dist='normal')
            gm = am.fit(disp='off', show_warning=False, cov_type='robust')

            last_sigma = float(gm.conditional_volatility.iloc[-1])
            fc_pt = np.array([arma_results[s]['preds'][i]
                              if s in arma_results and i < len(arma_results[s]['preds'])
                              else float(y_tr[-1]) for i in range(n_te)])
            sigma_arr = np.full(n_te, last_sigma)

            lo = fc_pt - Z80 * sigma_arr
            hi = fc_pt + Z80 * sigma_arr

            # Coverage (src/scoring.py)
            cov = pi_coverage(y_te, lo, hi)
            # Winkler score (src/scoring.py)
            winkler = winkler_score(y_te, lo, hi, alpha=0.20)
            # Pinball loss (src/scoring.py — proper scoring rule, Koenker & Bassett 1978)
            pb_lo  = pinball_loss(y_te, lo, tau=0.10)
            pb_hi  = pinball_loss(y_te, hi, tau=0.90)
            pb_avg = (pb_lo + pb_hi) / 2

            pi_rows.append({
                'series':         s,
                'coverage_80pct': round(cov, 3),
                'target':         0.80,
                'avg_width':      round(float(np.mean(hi - lo)), 6),
                'winkler':        round(winkler, 6),
                'pinball_avg':    round(pb_avg, 6),
            })
            flag = 'OK' if abs(cov - 0.80) < 0.08 else 'MISCALIBRATED'
            print(f'  {s:<15}: cov={cov:.3f}  Winkler={winkler:.5f}'
                  f'  Pinball={pb_avg:.5f}  {flag}')
        except Exception as e:
            print(f'  {s}: FAILED — {e}')

    pi_df = pd.DataFrame(pi_rows)
    print('\n=== Prediction Interval Scoring Summary ===')
    print(pi_df.to_string(index=False))
    pi_df.to_csv(OUTPUT_DIR / 'B_garch_pi.csv', index=False)
    print('Saved: output/B_garch_pi.csv')
    print('\nNote: Pinball is a proper scoring rule — cannot be gamed by widening intervals.')
    print('      Winkler and Pinball should agree directionally on model ranking.')

---
## 6. VAR Analysis — Cross-Parameter Dynamics

A VAR(p) is estimated on the **stationary system** to capture cross-parameter predictability.  

**Specification choice:**  
- I(1) series (α, η) → first-differenced: Δα, Δη  
- I(0) series (β, ρ, γ) → levels  
- This mixed system is valid if there are no cointegrating relationships among the stationary and  
  differenced series (tested in Section 7).  

**Lag selection:** BIC.  
**Baseline:** random walk y_{t-1} (NOT zero — the previous bug).  
**OOS:** rolling h=1, MSE ratio vs RW baseline.

In [ ]:
# Stationary system for VAR
I1_series  = [p for p in PARAMS if integration.get(p, 'I(0)') == 'I(1)']
I0_series  = [p for p in PARAMS if integration.get(p, 'I(0)') != 'I(1)']
VAR_COLS   = [f'd_{p}' for p in I1_series] + I0_series
# Fallback: if stationarity test inconclusive, use differenced alpha, eta + levels of rest
if not VAR_COLS:
    VAR_COLS = ['d_alpha', 'd_eta', 'beta', 'rho', 'gamma']
VAR_COLS = [c for c in VAR_COLS if c in df.columns]
print(f'VAR system: {VAR_COLS}')

var_data = df[['date'] + VAR_COLS].dropna().reset_index(drop=True)
n_var_split = int(len(var_data) * SPLIT)
var_train = var_data.iloc[:n_var_split][VAR_COLS].values
var_test  = var_data.iloc[n_var_split:][VAR_COLS].values

# BIC lag selection on training set
var_model_sel = VAR(var_train)
lag_order_res = var_model_sel.select_order(maxlags=10)
p_bic = lag_order_res.bic
if p_bic is None or p_bic < 1:
    p_bic = 1
print(f'VAR optimal lag (BIC): p={p_bic}')

# Fit on training set
var_fitted = VAR(var_train).fit(p_bic)
print(var_fitted.summary())

In [ ]:
# --- VAR OOS rolling forecast (correct RW baseline) ---
print('VAR rolling OOS h=1 (correct random-walk baseline)...')
N_var = len(var_data)
var_preds  = {c: [] for c in VAR_COLS}
var_actual = {c: [] for c in VAR_COLS}
var_baseline = {c: [] for c in VAR_COLS}

for i in range(N_var - n_var_split):
    y_tr = var_data.iloc[:n_var_split + i][VAR_COLS].values
    y_next = var_data.iloc[n_var_split + i][VAR_COLS].values  # actual
    y_last = var_data.iloc[n_var_split + i - 1][VAR_COLS].values  # RW baseline
    try:
        vm = VAR(y_tr).fit(p_bic, trend='c')
        fc = vm.forecast(y_tr[-p_bic:], steps=1)[0]
    except:
        fc = y_tr[-1]
    for j, c in enumerate(VAR_COLS):
        var_preds[c].append(fc[j])
        var_actual[c].append(y_next[j])
        var_baseline[c].append(y_last[j])  # correct RW

var_oos_rows = []
for c in VAR_COLS:
    a  = np.array(var_actual[c])
    p_ = np.array(var_preds[c])
    b  = np.array(var_baseline[c])
    mse_m = float(np.mean((a - p_) ** 2))
    mse_b = float(np.mean((a - b) ** 2))
    ratio = mse_m / mse_b if mse_b > 1e-15 else np.nan
    r2    = float(1 - mse_m / mse_b) if mse_b > 1e-15 else np.nan
    dm_s, dm_p = dm_test(a, p_, b, h=1)
    var_oos_rows.append({'series': c, 'MSE_ratio': round(ratio, 4),
                         'R2_OOS': round(r2, 4), 'DM_stat': dm_s, 'DM_p': dm_p})
    print(f'  {c:<15}: MSE-ratio={ratio:.4f}  R2={r2:.4f}  DM={dm_s}  p={dm_p}')

var_oos_df = pd.DataFrame(var_oos_rows)
var_oos_df.to_csv(OUTPUT_DIR / 'B_var_oos.csv', index=False)

# Granger causality within VAR
print('\n=== Granger causality test within VAR ===')
for causing in VAR_COLS:
    try:
        gc = var_fitted.test_causality(causing, VAR_COLS, kind='f')
        print(f'  {causing} causes all: F={gc.test_statistic:.3f}  p={gc.pvalue:.4f}')
    except:
        pass

---
## 7. Cointegration Analysis

With α and η classified as I(1), we test for cointegration between them.  
A cointegrating relationship would imply a long-run equilibrium between the  
ATM volatility level (α) and the smile curvature (η), consistent with the Heston SDE  
where the long-run variance level θ and vol-of-vol ξ are structurally linked.

**Tests used:**
- **Johansen trace test** (Johansen 1991): multivariate, H0 = rank ≤ r
- **Engle-Granger** (Engle & Granger 1987): bivariate residual ADF test
- **VECM** if cointegration is confirmed: estimates speed-of-adjustment coefficients

In [ ]:
from statsmodels.tsa.vector_ar.vecm import coint_johansen

# Test α-η cointegration (both I(1))
I1_cols = [p for p in PARAMS if integration.get(p, 'I(1)') == 'I(1)']
if not I1_cols:
    I1_cols = ['alpha', 'eta']  # fallback based on prior literature
print(f'Testing cointegration among I(1) series: {I1_cols}')

coint_data = df[I1_cols].dropna().values

# Johansen test
try:
    joh = coint_johansen(coint_data, det_order=0, k_ar_diff=1)
    print('\n=== Johansen Trace Test ===')
    print(f'  Trace statistics: {joh.lr1.round(4)}')
    print(f'  Critical values (5%): {joh.cvt[:, 1].round(4)}')
    rank_johansen = int(np.sum(joh.lr1 > joh.cvt[:, 1]))
    print(f'  Cointegration rank: {rank_johansen}')
except Exception as e:
    print(f'Johansen test failed: {e}')
    rank_johansen = 0

# Engle-Granger pairwise (for each pair of I(1) series)
print('\n=== Engle-Granger Pairwise Cointegration Tests ===')
from statsmodels.tsa.stattools import coint as eg_coint
for i, c1 in enumerate(I1_cols):
    for c2 in I1_cols[i + 1:]:
        sub = df[[c1, c2]].dropna()
        try:
            t_stat, p_val, crit = eg_coint(sub[c1], sub[c2], autolag='BIC')
            sig = '***' if p_val < 0.01 else '**' if p_val < 0.05 else '*' if p_val < 0.10 else ''
            print(f'  EG({c1}, {c2}): t={t_stat:.4f}  p={p_val:.4f} {sig}')
        except Exception as e:
            print(f'  EG({c1}, {c2}): FAILED — {e}')

In [ ]:
# --- VECM estimation (if cointegration confirmed) ---
if HAS_VECM and rank_johansen >= 1 and len(I1_cols) >= 2:
    print(f'\n=== VECM Estimation (rank={rank_johansen}) ===')
    vecm_data_tr = df_train[I1_cols].dropna().values
    vecm_data_te = df_test[I1_cols].dropna().values
    try:
        vecm_fit = VECM(vecm_data_tr, k_ar_diff=1, coint_rank=rank_johansen,
                        deterministic='co').fit()
        print('Loading matrix (speed of adjustment to long-run equilibrium):')
        alpha_mat = pd.DataFrame(vecm_fit.alpha, index=I1_cols)
        print(alpha_mat.round(6).to_string())
        print('\nCointegrating vector (beta):')
        print(vecm_fit.beta.round(6))

        # VECM OOS h=1
        vecm_fc = []
        n_vecm_tr = len(vecm_data_tr)
        full_vecm = np.vstack([vecm_data_tr, vecm_data_te])
        for i in range(len(vecm_data_te)):
            y_tr_v = full_vecm[:n_vecm_tr + i]
            try:
                vm_i = VECM(y_tr_v, k_ar_diff=1, coint_rank=rank_johansen,
                            deterministic='co').fit()
                fc = vm_i.predict(steps=1)[0]
            except:
                fc = y_tr_v[-1]
            vecm_fc.append(fc)

        vecm_fc = np.array(vecm_fc)
        for j, c in enumerate(I1_cols):
            actual_v   = vecm_data_te[:len(vecm_fc), j]
            baseline_v = np.concatenate([[vecm_data_tr[-1, j]], actual_v[:-1]])
            mse_m = float(np.mean((actual_v - vecm_fc[:, j]) ** 2))
            mse_b = float(np.mean((actual_v - baseline_v) ** 2))
            r2 = float(1 - mse_m / mse_b) if mse_b > 1e-15 else np.nan
            print(f'  VECM OOS {c}: R2={r2:.4f}  (positive = beats RW; negative = misspecified)')
    except Exception as e:
        print(f'VECM estimation failed: {e}')
else:
    print('No cointegration confirmed or rank=0 → VAR in levels is the correct specification.')

---
## 8. Rolling Johansen — Regime-Conditional Cointegration

**Research question:** Is α–η cointegration stable over time, or concentrated in specific market regimes?

**Setup:**
- 500-day sliding window over the full 2666-day sample
- Johansen trace test at 5% significance level in each window
- Window rank classification: 0 (no cointegration), 1 (one CV), 2 (full rank = both stationary)
- Regime comparison: calm vs stress windows (split at VIX Q75)

**Key finding from prior analysis:** rank=1 appears predominantly in stress regimes,  
suggesting the α–η cointegrating relationship is regime-conditional, not structural.

In [ ]:
WINDOW = 500
coint_series = I1_cols if len(I1_cols) >= 2 else ['alpha', 'eta']
roll_data    = df[coint_series].dropna().reset_index(drop=True)
N_roll = len(roll_data)

roll_ranks  = []
roll_dates  = []
for start in range(0, N_roll - WINDOW + 1, 5):  # step=5 for speed
    window_vals = roll_data.iloc[start:start + WINDOW].values
    try:
        joh_w = coint_johansen(window_vals, det_order=0, k_ar_diff=1)
        rank_w = int(np.sum(joh_w.lr1 > joh_w.cvt[:, 1]))
    except:
        rank_w = -1
    roll_ranks.append(rank_w)
    mid_idx = start + WINDOW // 2
    roll_dates.append(df['date'].iloc[min(mid_idx, len(df) - 1)] if 'date' in df.columns else mid_idx)

rank_series = pd.Series(roll_ranks)
total_w = len([r for r in roll_ranks if r >= 0])
print(f'=== Rolling Johansen ({WINDOW}-day window, step=5) ===')
for r_val in [0, 1, 2]:
    cnt = sum(1 for r in roll_ranks if r == r_val)
    pct = cnt / total_w * 100 if total_w > 0 else 0
    print(f'  Rank={r_val}: {cnt:4d} windows ({pct:.1f}%)')

# Regime-conditional EG test
if 'vix' in df.columns:
    vix_q75 = df['vix'].quantile(0.75)
    print(f'\nVIX Q75 = {vix_q75:.2f}')
    from statsmodels.tsa.stattools import coint as eg_coint
    for regime, mask in [('Calm (VIX<Q75)', df['vix'] < vix_q75),
                          ('Stress (VIX>Q75)', df['vix'] >= vix_q75)]:
        sub = df[mask][coint_series].dropna()
        if len(sub) < 50:
            continue
        try:
            t_s, p_v, _ = eg_coint(sub[coint_series[0]], sub[coint_series[1]], autolag='BIC')
            print(f'  EG {regime}: t={t_s:.4f}  p={p_v:.4f}')
        except Exception as e:
            print(f'  EG {regime}: failed — {e}')

# Plot rolling rank
fig, ax = plt.subplots(figsize=(13, 4))
ax.step(range(len(roll_ranks)), roll_ranks, where='mid', color='steelblue', lw=1.2)
ax.axhline(1, color='crimson', lw=1, ls='--', label='rank=1 (true cointegration)')
ax.set_ylabel('Johansen rank')
ax.set_xlabel('Window index')
ax.set_title(f'Rolling Johansen ({WINDOW}-day window) — {coint_series}')
ax.legend()
plt.savefig(PLOT_DIR / 'B_rolling_johansen.png', dpi=130, bbox_inches='tight')
plt.show()
print('Saved: figures/B_rolling_johansen.png')

---
## 9. PCA on the SSVI Surface + HAR on PC Scores

**Key insight:** SSVI parameter *changes* are near-white-noise (ARMA barely beats RW).  
The *levels* of the surface, however, are highly persistent and amenable to HAR-type forecasting.  

**Approach:**
1. Compute the full SSVI total-variance surface $\omega(k,T)$ on a 100-point grid (20k × 5T) for each date.
2. Apply PCA on the 2666 × 100 surface matrix. Fit on **train only** (anti-leakage).
3. Project test dates onto the train PCA basis.
4. Fit HAR on PC *level* scores: $s_i(t+h) = \beta_0 + \beta_1 s_i(t) + \beta_2 \bar{s}_i^{(5)}(t) + \beta_3 \bar{s}_i^{(22)}(t) + \varepsilon$
5. Reconstruct the surface from predicted scores and evaluate R²_OOS in $\omega$-space.

**Analogy:** The 4-PC structure (Level, Slope, Curvature, Skew) mirrors Nelson-Siegel yield curves  
(Diebold & Li 2006). This is not coincidental — both surfaces are driven by the same economic forces.

In [ ]:
def ssvi_omega(k, T, alpha, beta, rho, eta, gamma):
    """SSVI total implied variance w(k,T) = theta/2 * {1 + rho*phi*k + sqrt((phi*k+rho)^2+1-rho^2)}."""
    theta = np.exp(alpha) * (T ** beta)
    phi   = eta * theta ** (-gamma) / (1 + eta * theta ** (1 - gamma))
    disc  = np.maximum((phi * k + rho) ** 2 + (1 - rho ** 2), 0)
    return (theta / 2) * (1 + rho * phi * k + np.sqrt(disc))

# Surface grid
K_GRID = np.linspace(-0.40, 0.30, 20)
T_GRID = np.array([30, 60, 91, 182, 365]) / 365.0
KK, TT = np.meshgrid(K_GRID, T_GRID)
k_flat = KK.flatten()
t_flat = TT.flatten()
N_GRID = len(k_flat)  # 100
print(f'Surface grid: {len(K_GRID)} k-values x {len(T_GRID)} T-values = {N_GRID} points')

# Compute surface for each date
print('Computing SSVI surfaces...')
surf_rows = []
valid_mask = []
for _, row in df.iterrows():
    try:
        w = ssvi_omega(k_flat, t_flat, row['alpha'], row['beta'], row['rho'], row['eta'], row['gamma'])
        if np.all(np.isfinite(w)) and np.all(w > 0):
            surf_rows.append(w)
            valid_mask.append(True)
        else:
            surf_rows.append(np.full(N_GRID, np.nan))
            valid_mask.append(False)
    except:
        surf_rows.append(np.full(N_GRID, np.nan))
        valid_mask.append(False)

surf_mat  = np.array(surf_rows)  # (N_dates, 100)
valid_idx = np.where(valid_mask)[0]
surf_valid = surf_mat[valid_idx]
dates_valid = df['date'].iloc[valid_idx].values if 'date' in df.columns else valid_idx
print(f'Valid surfaces: {len(surf_valid):,} / {len(df):,}')

In [ ]:
# PCA fitted on TRAIN only
n_surf_train = int(len(surf_valid) * SPLIT)
surf_train = surf_valid[:n_surf_train]
surf_test  = surf_valid[n_surf_train:]

pca = PCA(n_components=min(10, N_GRID), random_state=RANDOM_STATE)
pca.fit(surf_train)

cum_var = np.cumsum(pca.explained_variance_ratio_)
N_PC    = int(np.searchsorted(cum_var, 0.99)) + 1
print(f'Components for 99% variance: N_PC = {N_PC}')
print('Variance explained per PC:')
for i in range(min(6, len(cum_var))):
    print(f'  PC{i+1}: {pca.explained_variance_ratio_[i]*100:.2f}%  (cumul: {cum_var[i]*100:.2f}%)')

# PC scores for train and test (projected onto train PCA)
pca_n = PCA(n_components=N_PC, random_state=RANDOM_STATE).fit(surf_train)
scores_train = pca_n.transform(surf_train)  # (n_train, N_PC)
scores_test  = pca_n.transform(surf_test)   # (n_test, N_PC)

print(f'\nNelson-Siegel analogy:')
if N_PC >= 1: print(f'  PC1 = Level (parallel shifts in total variance) — Var={pca.explained_variance_ratio_[0]*100:.2f}%')
if N_PC >= 2: print(f'  PC2 = Slope (term structure tilt) — Var={pca.explained_variance_ratio_[1]*100:.2f}%')
if N_PC >= 3: print(f'  PC3 = Curvature (smile width) — Var={pca.explained_variance_ratio_[2]*100:.2f}%')

In [ ]:
# HAR on PC scores — forecasting LEVELS, not differences
def har_features(scores, h):
    """Build HAR features for PC scores: s(t), avg_5(t), avg_22(t)."""
    df_s  = pd.DataFrame(scores, columns=[f'pc{i+1}' for i in range(scores.shape[1])])
    feats = {}
    for pc in df_s.columns:
        feats[f'{pc}_d']  = df_s[pc]
        feats[f'{pc}_w']  = df_s[pc].rolling(5,  min_periods=5).mean()
        feats[f'{pc}_m']  = df_s[pc].rolling(22, min_periods=22).mean()
    X = pd.DataFrame(feats).dropna()
    # Target: score at t+h
    Y = df_s.shift(-h).loc[X.index]
    valid = Y.notna().all(axis=1)
    return X[valid].values, Y[valid].values

print('HAR-PC forecasting on test set...')
pca_har_results = {}
all_scores = np.vstack([scores_train, scores_test])  # full series for HAR feature construction

for h in [1, 5, 20]:
    X_har, Y_har = har_features(all_scores, h)
    n_tr_har = len(X_har) - len(scores_test) + h
    if n_tr_har <= 0 or n_tr_har >= len(X_har):
        print(f'  h={h}: insufficient data')
        continue
    X_tr_h, Y_tr_h = X_har[:n_tr_har], Y_har[:n_tr_har]
    X_te_h, Y_te_h = X_har[n_tr_har:], Y_har[n_tr_har:]

    sc = StandardScaler().fit(X_tr_h)
    reg = LinearRegression().fit(sc.transform(X_tr_h), Y_tr_h)
    Y_pred = reg.predict(sc.transform(X_te_h))

    # Reconstruct surface and evaluate R2_OOS in omega-space
    surf_pred = pca_n.inverse_transform(Y_pred)
    surf_true = pca_n.inverse_transform(Y_te_h)
    # Naive baseline: last-known surface (persistence)
    surf_naive = pca_n.inverse_transform(all_scores[n_tr_har - h: n_tr_har - h + len(Y_te_h)])

    sse_m = float(np.mean((surf_true - surf_pred) ** 2))
    sse_n = float(np.mean((surf_true - surf_naive) ** 2))
    r2 = float(1 - sse_m / sse_n) if sse_n > 1e-15 else np.nan
    pca_har_results[h] = {'r2_oos': r2, 'sse_m': sse_m, 'sse_n': sse_n}
    print(f'  h={h:2d}: R2_OOS = {r2:.4f}  (HAR-PC vs naive persistence in omega-space)')

pca_har_df = pd.DataFrame({'horizon': list(pca_har_results.keys()),
                            'R2_OOS': [v['r2_oos'] for v in pca_har_results.values()]})
pca_har_df.to_csv(OUTPUT_DIR / 'B_pca_har_results.csv', index=False)
print('Saved: output/B_pca_har_results.csv')

---
## 10. Comprehensive Results Summary

Best model per parameter per forecasting family, ranked by OOS MSE ratio vs random-walk baseline.

In [ ]:
summary_rows = []
for s in ALL_SERIES:
    arma_r  = arma_results.get(s, {})
    armax_r = armax_results.get(s, {}) if armax_results else {}
    best_model  = 'ARMA'
    best_ratio  = arma_r.get('ratio', np.nan)
    best_r2     = arma_r.get('r2_oos', np.nan)
    if armax_r and armax_r.get('ratio', 1.0) < best_ratio:
        best_model = 'ARMAX'
        best_ratio = armax_r['ratio']
        best_r2    = armax_r.get('r2_oos', np.nan)
    i_order = integration.get(s, integration.get(s.replace('d_', ''), 'I(0)'))
    summary_rows.append({
        'series':           s,
        'I_order':          i_order,
        'ARMA_order':       str(arma_orders.get(s, '?')),
        'ARMA_ratio':       round(arma_r.get('ratio', np.nan), 4),
        'ARMA_R2':          round(arma_r.get('r2_oos', np.nan), 4),
        'ARMA_LB_ok':       arma_r.get('ok', False),
        'ARMAX_ratio':      round(armax_r.get('ratio', np.nan), 4),
        'ARMAX_R2':         round(armax_r.get('r2_oos', np.nan), 4),
        'best_model':       best_model,
        'best_ratio':       round(best_ratio, 4),
        'beats_RW':         best_ratio < 1.0 if not np.isnan(best_ratio) else False
    })

summary_df = pd.DataFrame(summary_rows)
print('=== FINAL SUMMARY: Best Model per Series (h=1 OOS) ===')
print(summary_df.to_string(index=False))
summary_df.to_csv(OUTPUT_DIR / 'B_final_summary.csv', index=False)
print('\nSaved: output/B_final_summary.csv')

# PCA-HAR summary
print('\n=== PCA-HAR Surface Forecasting (R2_OOS vs naive persistence) ===')
for h, res in pca_har_results.items():
    print(f'  h={h:2d}: R2_OOS = {res["r2_oos"]:.4f}')

print('\n=== VAR OOS Results (corrected RW baseline) ===')
if 'var_oos_df' in dir():
    print(var_oos_df.to_string(index=False))

print('\nNote: Δα, Δη near-white-noise → ARMA MSE-ratio ≈ 1. This is a correct empirical result,')
print('not a model failure. Level parameters (ρ, γ) are more forecastable.')